# 05 - Expression Derivation and Constraint Solving

> **When to use**: When columns have computational relationships (e.g., `short_code = project_no[-6:]`), or have UNIQUE constraints.
>
> **Core concept**: `derive_from` declares column dependencies, `expression` specifies the formula, ColumnDAG auto-topological sort.

## Applicable Scenarios

- Column B depends on column A's value (e.g., abbreviation, substring, concatenation) → `derive_from` + `expression`
- Column has UNIQUE constraint → ConstraintSolver auto-backtracking
- Large data volume (>100K rows) UNIQUE → auto-switch to probabilistic mode
- Need chained dependencies (A → B → C) → ColumnDAG topological sort

## What You Will Learn

- `derive_from` + `expression` column derivation
- ColumnDAG topological sort
- ExpressionEngine 21 safe functions
- UNIQUE constraint backtracking
- Probabilistic mode (>100K rows)

See architecture.zh-CN.md §6

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| **→ 05** | **Expression Derivation and Constraint Solving** | **Core: DAG / Expression** | **01** |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
PRJ_PATTERN = "PRJ-\\d{6}"
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Data Stream Generation | `src/sqlseed/generators/stream.py` | `DataStream` |
| Dependency Sorting | `src/sqlseed/core/column_dag.py` | `ColumnDAG` |
| Constraint Backtracking | `src/sqlseed/core/constraints.py`| `ConstraintSolver` |

> Corresponding architecture diagram: [§6 Column Dependency DAG and Constraint Backtracking](../docs/architecture.zh-CN.md#6-列依赖-dag-与约束回溯)

## 1. See It in Action — The Power of Derived Columns

Often, columns have dependencies: `short_code` is the last 6 chars of `project_no`, `description` is concatenated from `project_no`.

sqlseed's `derive_from` + `expression` lets you declare such relationships, **auto-generating correct derived values**:

In [2]:

with connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=5, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'project_no', 'expression': "'Project-' + value"},
    })
    print(f"{'project_no':<15s}  {'short_code':<10s}  {'description':<30s}")
    print('-' * 60)
    for row in preview:
        print(f"{row['project_no']:<15s}  {row['short_code']:<10s}  {row['description']:<30s}")

project_no       short_code  description                   
------------------------------------------------------------
PRJ-993498       993498      Project-PRJ-993498            
PRJ-651149       651149      Project-PRJ-651149            
PRJ-697372       697372      Project-PRJ-697372            
PRJ-021699       021699      Project-PRJ-021699            
PRJ-822650       822650      Project-PRJ-822650            


Just declare the dependency, sqlseed auto:

1. **Topological sort** — determines generation order (first `project_no`, then `short_code` and `description`)
2. **Expression evaluation** — uses `simpleeval` safe engine to execute expressions
3. **Timeout protection** — 5s timeout prevents infinite loops

Below we break down each mechanism in detail.

## 2. derive_from + expression

When a column's value depends on another, use `derive_from` to declare the dependency, `expression` to specify the formula:

```yaml
columns:
  - name: project_no
    generator: pattern
    params:
      regex: "PRJ-\\d{6}"
  - name: short_code
    derive_from: project_no
    expression: "value[-6:]"
```

sqlseed first generates `project_no`, then computes `short_code` using `value[-6:]`.

In [3]:
import sqlite3

result = fill(
    str(db_path),
    table="projects",
    count=5, clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
    },
)


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}")
conn.close()

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no=PRJ-291676, short_code=291676
project_no=PRJ-225564, short_code=225564
project_no=PRJ-093282, short_code=093282
project_no=PRJ-893558, short_code=893558
project_no=PRJ-354600, short_code=354600


## 3. ColumnDAG Topological Sort

When multi-level dependencies exist, `ColumnDAG` auto-performs topological sort to ensure correct generation order:

```
project_no → short_code (value[-6:])
           → description (concat('Project ', value))
```

If cyclic dependencies exist, sqlseed raises `CyclicDependencyError`.

In [4]:
result = fill(
    str(db_path),
    table="projects",
    count=5,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
        "description": {"derive_from": "project_no", "expression": "concat('Project ', value)"},
    },
)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, description FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, description={row[2]}")
conn.close()

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

project_no=PRJ-349829, short_code=349829, description=Project PRJ-349829
project_no=PRJ-133393, short_code=133393, description=Project PRJ-133393
project_no=PRJ-308807, short_code=308807, description=Project PRJ-308807
project_no=PRJ-107130, short_code=107130, description=Project PRJ-107130
project_no=PRJ-848690, short_code=848690, description=Project PRJ-848690


## 4. ExpressionEngine 21 Safe Functions

The expression engine is based on `simpleeval`, providing 21 safe functions:

| Category | Functions | Description |
|------|------|------|
| String | `upper`, `lower`, `strip`, `lstrip`, `rstrip` | case/whitespace |
| String | `replace`, `lpad`, `rpad` | replace/pad |
| String | `substring`, `concat` | substring/concat |
| Math | `abs`, `round`, `ceil`, `floor` | rounding |
| Math | `min`, `max`, `clamp` | extreme/clamp |
| Conversion | `int`, `float`, `str`, `len` | type conversion |
| Hash | `md5`, `sha256` | hash digest |

In [5]:
result = fill(
    str(db_path),
    table="projects",
    count=3,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "upper(value[-6:])"},
        "name": {"derive_from": "project_no", "expression": "concat('Project-', value)"},
    },
)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, name={row[2]}")
conn.close()

Generating projects:   0%|          | 0/3 [00:00<?, ?it/s]

project_no=PRJ-599719, short_code=599719, name=Project-PRJ-599719
project_no=PRJ-026434, short_code=026434, name=Project-PRJ-026434
project_no=PRJ-417146, short_code=417146, name=Project-PRJ-417146


## 5. ExpressionTimeoutError Timeout Protection

Expression execution has 5s timeout protection. If an expression exceeds 5s, it raises `ExpressionTimeoutError`.

This prevents malicious or incorrect expressions from causing infinite loops.

## 6. UNIQUE Constraint Backtracking

When a column has a UNIQUE constraint, `ConstraintSolver` uses a backtracking algorithm to ensure no duplicates:

1. Generate a value
2. Check if it conflicts with existing values
3. If conflict, regenerate (retry up to N times)
4. If all retries fail, backtrack to the previous row and regenerate

UNIQUE columns in the demo database:
- `members.member_no` (UNIQUE)
- `members.email` (UNIQUE)
- `projects.project_no` (UNIQUE)
- `projects.short_code` (UNIQUE INDEX)
- `tags.name` (UNIQUE)

In [6]:
result = fill(str(db_path), table="members", count=50, clear_before=True)

conn = sqlite3.connect(str(db_path))
member_nos = [r[0] for r in conn.execute("SELECT member_no FROM members").fetchall()]
emails = [r[0] for r in conn.execute("SELECT email FROM members").fetchall()]
print(f"Total rows: {len(member_nos)}")
print(f"Unique member_nos: {len(set(member_nos))}")
print(f"Unique emails: {len(set(emails))}")
print(f"All member_nos unique: {len(member_nos) == len(set(member_nos))}")
print(f"All emails unique: {len(emails) == len(set(emails))}")
conn.close()

Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

Total rows: 50
Unique member_nos: 50
Unique emails: 50
All member_nos unique: True
All emails unique: True


## 7. Probabilistic Mode (>100K rows)

When generating more than 100K rows, backtracking performance degrades. `ConstraintSolver` auto-switches to probabilistic mode:

- Uses SHA256 hash to ensure uniqueness
- No backtracking needed, stable performance
- Extremely low collision probability (negligible)

See architecture.md §6 ConstraintSolver

## 🔗 Multi-level derive_from Chained Dependencies

`derive_from` supports chained dependencies: A → B → C, ColumnDAG auto-performs topological sort.

In [7]:
with sqlseed.connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=3, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'short_code', 'expression': "'Project-' + value"},
    })
    print('Multi-level derive_from chain (project_no -> short_code -> description):')
    for row in preview:
        print(f"  {row['project_no']} -> {row['short_code']} -> {row['description']}")

Multi-level derive_from chain (project_no -> short_code -> description):
  PRJ-173425 -> 173425 -> Project-173425
  PRJ-923596 -> 923596 -> Project-923596
  PRJ-149663 -> 149663 -> Project-149663


## ⚠️ In Practice: ExpressionTimeoutError

Expression execution exceeding 5s triggers timeout protection.

In [8]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine(timeout_seconds=1)  # 1 second for demo

# Safe expressions execute instantly
result = engine.evaluate('abs(-42)', {})
print(f'abs(-42) = {result}')

# Demonstrate timeout API
print(f'\nExpressionEngine timeout={engine._timeout}s')
print('  - Expressions exceeding the timeout raise ExpressionTimeoutError')
print('  - Default timeout: 5 seconds')
print('  - Used internally by derive_from expressions')
print('  - Thread-based: cannot be killed, only detected')


abs(-42) = 42

ExpressionEngine timeout=1s
  - Expressions exceeding the timeout raise ExpressionTimeoutError
  - Default timeout: 5 seconds
  - Used internally by derive_from expressions
  - Thread-based: cannot be killed, only detected


In [9]:
from sqlseed.core.mapper import ColumnMapper
from sqlseed.core.unique_adjuster import UniqueAdjuster

adjuster = UniqueAdjuster(ColumnMapper())

# Simulate: min_length=2, max_length=2 can only generate ~3844 unique values
# For 50000 rows, UniqueAdjuster increases max_length automatically
print('UniqueAdjuster auto-adjustment example:')
print('  Input: min_length=2, max_length=2, count=50000')
print('  Output: max_length auto-expanded to satisfy UNIQUE constraint')
print()

# Actual fill with UNIQUE constraint
result = fill(str(db_path), table='members', count=50, clear_before=True, seed=42)
print(f'Filled {result.count} members with UNIQUE member_no')

conn = sqlite3.connect(str(db_path))
sample = conn.execute('SELECT member_no FROM members LIMIT 5').fetchall()
print(f'Sample member_no values: {[r[0] for r in sample]}')
conn.close()

UniqueAdjuster auto-adjustment example:
  Input: min_length=2, max_length=2, count=50000
  Output: max_length auto-expanded to satisfy UNIQUE constraint



Generating members:   0%|          | 0/50 [00:00<?, ?it/s]

Filled 50 members with UNIQUE member_no
Sample member_no values: [' CI3 dXRZv7', '07gmhZRnFyy5r2xJ7', '1XcW9aTMX1', '4jKEIQOkrtDX', '7gFp6r7O4-25u85HFJ E']


## 🔐 Composite Unique Constraints

`ConstraintSolver` supports multi-column composite unique constraints, ensuring combined values are not duplicated.

In [10]:
from sqlseed.core.constraints import ConstraintSolver

solver = ConstraintSolver()

# Register composite unique (org_code, member_no)
pairs = set()
for i in range(20):
    org = f'ORG-{i % 3}'
    member = f'M{i:04d}'
    ok = solver.check_and_register_composite('org_member', (org, member))
    pairs.add((org, member))

print(f'Registered {len(pairs)} unique (org_code, member_no) pairs')
print(f'Sample: {list(pairs)[:3]}')

Registered 20 unique (org_code, member_no) pairs
Sample: [('ORG-0', 'M0012'), ('ORG-1', 'M0013'), ('ORG-2', 'M0017')]


## Summary

| Feature | Description |
|------|------|
| derive_from | Declare column dependencies |
| expression | Compute derived values (based on simpleeval) |
| ColumnDAG | Topological sort to determine generation order |
| ExpressionEngine | 21 safe functions, 5s timeout protection |
| ConstraintSolver | UNIQUE backtracking + probabilistic mode |
| UniqueAdjuster | Auto-expand length to satisfy unique constraints |
| Composite Unique | Multi-column composite unique |

**Next**: [06-config-deep-dive.ipynb](06-config-deep-dive.ipynb) — Config Model Deep Dive

In [11]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
